# 

# Red-team referee report — round 1

## Assessment

The arithmetic identities in the headline are mostly reproducible, but that is not the same as validating the headline: `results/summary.json` does yield a 4.909 benefit–cost median (2.067–11.041), the stance endpoints yield about 45.5×, the raw location calculation yields \$3.566 billion (20.4%), and the cause table yields about \$9,477/QALY at its favorable end. The paper nevertheless presents a highly assumption-driven scenario as an estimate of realized portfolio impact. Its dominant result is created by treating organization-level service-location labels as delivery shares and assigning every routed non-U.S. health/cash/food dollar—including high-income-country and unattributable activity—the same exceptionally favorable “global health” prior. No recipient outcomes or use-of-gift data validate that move. The geographic audit is much weaker than advertised, the frontier counterfactual extrapolates far beyond observed room for funding, the novelty claim lacks a defensible literature search, and the canonical rendered paper is stale relative to source edits despite claiming prose cannot drift. I would not post the paper in its current empirical form; it could become a useful transparent scenario exercise only after the causal and geographic claims are radically narrowed.

## Findings

1.  **SEVERITY: CRITICAL**  
    **LOCATION:** Abstract, §4.1, and conclusion: “the 4.8% of dollars funding health and development delivered outside the United States produces 69% of estimated health impact”; “most of that health comes from the twentieth of her dollars delivered in low-income countries.”  
    **Problem:** The 69.1% is generated by a routing rule, not learned from outcomes. `src/msqaly/allocation.py` sends the non-U.S.-location fraction of four broad targets (`SPLIT_HEALTH`, mental health, food, and cash) to one `global_health` archetype, whose unsupported 90% prior is \$300–\$4,000/QALY. “Non-U.S.” is then silently treated as “LMIC global health.” The model’s own regional matrix assigns the routed bucket 17.6% to unattributable “global,” 3.4% to Europe/Central Asia, 0.16% to Canada, and additional shares to regions containing high-income countries. Crisis Text Line activity in Europe, for example, is not malaria prevention. The code applies the same prior to heterogeneous organizations and interventions without estimating their mix. Thus 4.8% → 69.1%, the ~\$10,000 realized ratio, and the tripling in the correction history are chiefly consequences of a favorable prior attached to a coarse label.  
    **Concrete fix:** Do not call this an impact estimate until the routed dollars are stratified by recipient, intervention, country/income group, and actual use of Scott’s gift. Apply GiveWell-like anchors only to identified comparable programs; retain high-income and unattributable work under appropriate anchors. Until then, report the 69.1% solely as a named scenario (“if all routed non-U.S. dollars have this stipulated prior”), add a no-geographic-repricing baseline, and place substantial probability on no effect and harm.

2.  **SEVERITY: CRITICAL**  
    **LOCATION:** Introduction and limitations: “The estimates here are a lower bound on total wellbeing produced”; “Credibility shrinks toward zero but never below it, so the model places no mass on harmful giving.”  
    **Problem:** This is not a lower bound. Omitting non-health outcomes makes the calculation partial, not conservative: omitted outcomes can be negative, included interventions can harm health, and unrestricted grants can displace other spending. Every archetype is forced to have a positive effect, including assumption-only categories, so the model structurally produces benefits even where it says evidence is absent. The 4.9 benefit–cost result is therefore upward constrained, not lower-bounded.  
    **Concrete fix:** Replace “lower bound” with “partial health-only accounting under positive-effect priors.” Add explicit point mass at zero and plausible negative-effect support, plus a structural scenario in which unsupported archetypes receive zero credit. Do not describe the output as realized impact without recipient-level outcome evidence.

3.  **SEVERITY: MAJOR**  
    **LOCATION:** §2.3: “an estimated 20% of disclosed dollars (\$3.57 billion of \$17.46B) goes to organizations serving abroad.”  
    **Problem:** The typed number is mechanically correct only for the pre-audit rule under which a bare `global` label counts as 100% non-U.S.: \$3.56623B / \$17.460132B = 20.425%. The very next paragraph explains why that rule is wrong. Applying the repository’s own audited `us_share` overlay to the same disclosed gifts gives about \$3.343B, or 19.15%, not \$3.57B. More importantly, a recipient listing a service location does not show that Scott’s gift “goes” there, and equal-count weighting of locations is not expenditure weighting.  
    **Concrete fix:** Replace the headline with the audited figure, identify it as “dollars to organizations listing non-U.S. service locations under an equal-location proxy,” and provide an uncertainty interval or bounds. Never use “goes to,” “delivered,” or “receives” for a location without expenditure or grant-use data.

4.  **SEVERITY: MAJOR**  
    **LOCATION:** §2.3 and Table 2 caption: “An audit of the 50 largest organizations reporting only \[‘global’\]”; “fractions taken from each organization’s own filings and program reports, independently verified by a second model family.”  
    **Problem:** This description overstates the audit. Twenty of the 50 organizations do not report only `global`; they also have U.S. or regional location labels. The audit file contains only 16 high-confidence rows, versus 17 medium and 17 low; the second model marks 16 rows “refute” and one “adjust,” not “verify.” Fully \$394.8M of the \$1.089B audited global slice is in rows the second model refuted. Several low-confidence rows explicitly say no geographic spending split exists, yet assign exact fractions anyway. In 46 of 50 cases the reconciliation is simply the mean of two model outputs. Those are heuristic judgments, not audited spending shares.  
    **Concrete fix:** Describe this as a dual-model coding exercise, publish a validation rule against observed financial shares, and separate measured expense shares from project counts, staff counts, and guesses. Report results by confidence tier and under each model separately. Reserve “verified” for rows confirmed from a common, contemporaneous spending denominator.

5.  **SEVERITY: MAJOR**  
    **LOCATION:** §2.3: “Planned Parenthood and Crisis Text Line … are roughly 95% and 93% U.S. by their own reporting.”  
    **Problem:** The prose does match the scalar fields in `geo_audit.jsonl` (`us_share` 0.955 and 0.9295), so these are not fabricated. But the use is temporally and internally unstable. Scott’s gifts were in 2022 (Planned Parenthood) and 2020 (Crisis Text Line), whereas the audit relies on FY2025 and 2024 reporting. Planned Parenthood Global was sunset in FY2025, making the later share especially poor evidence for deployment of a 2022 gift. The same rows contain different U.S. regional shares (0.9500 and 0.9376), so the scalar allocation and regional decomposition do not use identical fractions. Neither organization reports how Scott’s unrestricted gift itself was divided.  
    **Concrete fix:** Use the closest grant-period audited statements and distinguish total organizational expenses from Scott-gift deployment. Reconcile `us_share` and `region_shares` to one denominator with a schema test. If historical allocation is unavailable, give a range and label the 95%/93% figures as later-period organization-wide proxies.

6.  **SEVERITY: MAJOR**  
    **LOCATION:** §4.2: “roughly 521×”; “105.5 million QALYs”; “same realization and credibility discounts”; “a floor plausibly under 20–40×”; “remains large under any plausible account of diminishing returns.”  
    **Problem:** The 521 and 105.5M values reproduce the code, but the interpretation does not. The \$175–\$241 conversion is model arithmetic—\$4,000–\$5,500 per life, inflated by an assumed 2023 price level and divided by 25.1 stipulated discounted QALYs—not a published GiveWell QALY estimate. GiveWell describes these figures as retrospective 2022–2024 weighted averages, warns that marginal opportunities differ, and explicitly analyzes funding opportunities rather than whole organizations ([GiveWell](https://www.givewell.org/impact-estimates)). The model does not apply the “same” credibility discount: the frontier receives an RCT-grade mean of 0.85 while portfolio archetypes receive different tier means. Multiplying a historical marginal price by \$30.3B creates a knowingly unattainable 105.5M figure. The cited cash-transfer post reports a mixed consumption-and-health benchmark and does not state a 20–40× QALY floor; deriving such a range from its mortality example requires additional contestable arithmetic ([GiveWell cash review](https://blog.givewell.org/2024/11/12/re-evaluating-the-impact-of-unconditional-cash-transfers/)). “Any plausible account” is untested. “Hundreds of millions” is supportable—GiveWell says it directs several hundred million dollars per year—but it underscores that tens of billions is outside the evidence.  
    **Concrete fix:** Delete the full-portfolio 105.5M counterfactual. Report only a marginal per-dollar benchmark and finite opportunity tiers with explicit capacity. Say “the same discounting framework,” not “the same discounts.” Show the full derivation and uncertainty for any cash/QALY backstop, and remove universal language about diminishing returns.

7.  **SEVERITY: MAJOR**  
    **LOCATION:** §4.1: “a realized cost of about \$10,000 per QALY against \$350,000–\$450,000 for the large U.S. buckets”; “The 1,200-fold spread in realized cost-effectiveness … restates … \[an\] empirical claim.”  
    **Problem:** About \$10,000 is a faithful rounding of the model’s \$9,477 output. The exact modeled max/min across Table 1 is about 1,278× (\$12.109M / \$9,477), so “1,200-fold” is also a tolerable rough number. But it is not an empirical spread in realized portfolio performance. Its expensive endpoint is arts, whose QALYs are generated by an assumption-only prior and a near-zero credibility input; the cheap endpoint is the unsupported routing described above. The \$350k–\$450k phrase does not describe the adjacent cause table’s large U.S. causes (e.g. education ~\$301k, workforce ~\$612k, equity ~\$3M); it selectively resembles later U.S. geographic rows, one of which is ~\$455k.  
    **Concrete fix:** Call these modeled ratios under stipulated priors, identify the exact endpoints, and move the geographic range next to Table 2 if retained. Replace the 1,200-fold rhetorical claim with comparisons among genuinely evidenced health interventions and separately disclose how much of the spread is mechanically induced by credibility-tier inputs.

8.  **SEVERITY: MAJOR**  
    **LOCATION:** Introduction: “the first portfolio-level cost-effectiveness estimate for a major philanthropist”; “no published estimate prices the portfolio in outcomes of any kind.”  
    **Problem:** No search protocol or review supports this priority claim, and adjacent prior art is substantial. GiveWell aggregates \$1.45B of directed funding into an estimated 340,000 lives saved; Robin Hood has long used a common benefit–cost metric across grant programs; the Constellation Fund reports an aggregate portfolio BCR; and CEGA describes portfolio-level social-rate-of-return analysis for a donor’s Development Innovation Ventures portfolio ([GiveWell](https://www.givewell.org/impact-estimates), [Robin Hood](https://robinhood.org/our-work/our-approach/), [Constellation Fund](https://constellationfund.org/impact-news-insights/investment-report-2025/), [CEGA](https://cega.berkeley.edu/article/donors-can-spend-more-money-better-with-cost-evidence/)). Whether the author can define “major philanthropist” narrowly enough to exclude all of these is not evidence of priority.  
    **Concrete fix:** Remove “first,” or narrow it to “the first public QALY scenario analysis I found for Scott’s disclosed lifetime portfolio,” document databases, terms, dates, and inclusion criteria, and discuss the closest prior portfolio analyses.

9.  **SEVERITY: MAJOR**  
    **LOCATION:** Abstract and opening sentence: “\$26.39 billion … since 2019 — roughly a third of American megagiving in 2025”; “gave away \$7 billion in 2025 … about a third of every megagift.”  
    **Problem:** The underlying comparison is real but the paper splices incompatible numerators. Giving USA reporting puts 2025 megagifts at \$19.2B and uses \$6.65B for Scott’s contributions in that comparison, about 34.6%; it defines a 2025 megagift as at least roughly \$600M ([AP summary of Giving USA](https://apnews.com/article/8363b76bc8cf854f6865c31129e8a4b1)). The paper instead juxtaposes Scott’s Yield Giving total of \$7.166B (37.3% of \$19.2B), without explaining the \$516M accounting difference. The abstract’s grammar can be read as comparing the \$26.39B lifetime total with 2025 megagiving, which is plainly not one third. “A third of every megagift” is also not the same as one third of aggregate megagift dollars.  
    **Concrete fix:** State the sourced ratio exactly: “Giving USA counts \$6.65B of Scott’s 2025 contributions against \$19.2B of U.S. megagift dollars (34.6%).” Define the threshold and explain the difference from Yield Giving’s \$7.166B before using either total.

10. **SEVERITY: MAJOR**  
    **LOCATION:** Abstract and §5.1: “restricting to randomized evidence”; “taking every cited effect at face value”; “a 45× range that dwarfs every other modeling choice.”  
    **Problem:** The 45× ratio is numerically reproducible (414,264 / 9,114 = 45.46), but the endpoint labels are false literally. “RCT-only” gives every non-randomized tier a positive credibility mean of 0.01, not zero; “face value” uses 0.999, not 1. More seriously, “every other modeling choice” has not been established by a comprehensive multiverse or global sensitivity analysis. The tornado reports one set of rank correlations inside the default model; it cannot rule out structural choices such as the geographic routing, which itself tripled the headline.  
    **Concrete fix:** Rename the endpoints “near-RCT-only floor” and “near-face-value,” state the 0.01/0.999 definitions, and limit the conclusion to “larger than any one-dimensional distributional variation tested.” Add structural scenarios for routing, zero/harm, alternative archetype mappings, and imputation.

11. **SEVERITY: MAJOR**  
    **LOCATION:** §6: “Sync tests force … this manuscript’s inputs to equal fresh derivations … so prose and model cannot drift apart.”  
    **Problem:** The canonical `paper/review/rendered.txt` has already drifted from `paper/index.qmd`. The rendered paper says “documents extreme,” “central fact,” “floor plausibly under 20–40×,” “the honest statement,” and “any plausible account”; the tracked source now softens or qualifies these phrases. It also adds “to my knowledge” and narrows the claim that the method applies to any giver. The priority numbers 4.9, 2.1–11.0, 45×, \$3.57B, 20%, \$175–\$241, \$10k, \$350k–\$450k, and 1,200-fold are typed prose rather than computed interpolations. Existing sync tests therefore do not deliver the stated guarantee.  
    **Concrete fix:** Re-render the canonical manuscript before circulation. Interpolate every model-derived number, create tests for externally typed constants and their denominators, and fail CI when the rendered text/PDF is older than or semantically inconsistent with `index.qmd` and the result artifacts.

12. **SEVERITY: MAJOR**  
    **LOCATION:** §4 Results: “the health alone repays the giving 4.9 times over (90% interval 2.1–11.0).”  
    **Problem:** These figures match `summary.json`, but “repays” implies a fiscal or cash return. The numerator is a simulated QALY count multiplied by an uncertain HHS value per QALY used for regulatory benefit valuation; it is not money returned to government, Scott, recipients, or society. HHS presents these as default inputs for regulatory impact analysis, not a measured willingness to pay for this portfolio ([HHS ASPE](https://aspe.hhs.gov/reports/standard-ria-values)). The prose also calls a triangular sensitivity distribution “the U.S. government’s value,” singular.  
    **Concrete fix:** Say “the median monetized health-benefit/cost ratio is 4.9 under the HHS regulatory valuation distribution.” Report the HHS low/central/high values and keep this separate from fiscal return language.

13. **SEVERITY: MINOR**  
    **LOCATION:** §5.2–5.3: “moved … the median from about 205,000 to 201,923”; “roughly 98,000 to 87,000”; “roughly 70,000”; “tripled … to roughly 205,000”; “eleven public revision rounds”; “none was caught by the author unaided.”  
    **Problem:** The numerical trajectory is broadly supported by Git: 97,527 → 86,716; 70,517/69,696; 203,949; 201,923. The 203,949 → 201,923 audit change is about 1.0%, and the blended ratio moved \$148,577 → \$150,067. However, “eleven rounds” has no stated counting rule, and the claims about who detected each error are not evidenced by a revision table. A commit history demonstrates changes, not discovery provenance.  
    **Concrete fix:** Add a correction table with commit/tag, prior and new result, exact changed assumption, detector, independent verifier, and evidence link. Define whether a “round” means a unique numerical state, a review session, a commit, or a release.

14. **SEVERITY: MINOR**  
    **LOCATION:** §2.1 and §6: “inflates each tranche to 2026 dollars with CPI-U, so dollars and cost-effectiveness evidence share one price level”; “every value carries its … dollar vintage.”  
    **Problem:** The \$30.302B total correctly applies the stored tranche factors, and BLS confirms that the 2025 annual average is based on 11 published months because October was unavailable ([BLS](https://www.bls.gov/cpi/additional-resources/2025-federal-government-shutdown-impact-cpi-faq.htm)). But “2026 dollars” actually means May 2026 CPI-U (335.123), even though June 2026 (333.952) was available before the paper date ([FRED](https://fred.stlouisfed.org/series/CPIAUCNS)). The 2022 tranche includes gifts made since June 2021, so disclosure year is not gift year. More importantly, the allocation pipeline derives cause shares in nominal dollars and then multiplies them by the real \$30.302B total; it does not inflate each disclosed/imputed gift before deriving real-dollar cause shares. An approximate reconstruction changes the global-health share from 4.818% nominal to about 4.795% real (small, but contrary to the prose), with larger movement for environment (~0.41 percentage point).  
    **Concrete fix:** Call the price level “May 2026 dollars,” explain disclosure-year booking, and either update to a fixed justified reference month or use the annual average when available. Inflate gift/imputed amounts before deriving cause shares, then regenerate all allocations and results.

15. **SEVERITY: MINOR**  
    **LOCATION:** §2.2: “The imputed third moves each cause share by at most about 1.5 percentage points — the disclosed two-thirds is representative.”  
    **Problem:** The 1.5-point maximum is reproducible under this imputation, but the conclusion is circular. The elasticity is fit only on 1,313 disclosed gift–filing pairs, and 287 of 676 undisclosed gifts lack usable pre-gift revenue and receive a window-median weight. Disclosure and 990 matching are selective, especially for foreign recipients. Similarity between disclosed-only shares and shares produced by a model trained on disclosed gifts does not establish external representativeness.  
    **Concrete fix:** Say “under this imputation model.” Report standard errors and diagnostics for the elasticity, disclosure/match selection by year and geography, alternative imputation models, and bounds assigning unmatched foreign gifts to plausible cause/geography extremes.

RECOMMENDATION: Reject